In [1]:

import pandas as pd
import numpy as np

from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Load datasets
data_dir = Path("../data/processed")

products_df = pd.read_csv(
    data_dir / "product_info_cleaned.csv",
    dtype={"product_id": str}
)

train_df = pd.read_csv(
    data_dir / "train_interactions.csv",
    dtype={"author_id": str, "product_id": str}
)

test_df = pd.read_csv(
    data_dir / "test_interactions.csv",
    dtype={"author_id": str, "product_id": str}
)

print("MODEL OPTIMIZATION")
print("-" * 40)

print("Products:", len(products_df))
print("Training interactions:", len(train_df))
print("Testing interactions:", len(test_df))

print("\nMissing combined features:",
      products_df["combined_features"].isna().sum())


MODEL OPTIMIZATION
----------------------------------------
Products: 8494
Training interactions: 213982
Testing interactions: 66871

Missing combined features: 0


In [2]:

# Baseline TF-IDF Model

tfidf_baseline = TfidfVectorizer(
    stop_words="english",
    max_features=10000,
    min_df=2,
    ngram_range=(1, 2),
    dtype=np.float32
)

baseline_matrix = tfidf_baseline.fit_transform(
    products_df["combined_features"].fillna("")
)

product_indices = pd.Series(
    products_df.index,
    index=products_df["product_id"]
)

print("BASELINE MODEL")
print("-" * 40)

print("TF-IDF Matrix Shape:", baseline_matrix.shape)
print("Vocabulary Size:", len(tfidf_baseline.vocabulary_))
print("Non-zero Values:", baseline_matrix.nnz)


BASELINE MODEL
----------------------------------------
TF-IDF Matrix Shape: (8494, 10000)
Vocabulary Size: 10000
Non-zero Values: 1062856


In [3]:

# Fixed Evaluation Users

common_users = pd.Index(
    train_df["author_id"].unique()
).intersection(
    test_df["author_id"].unique()
)

evaluation_users = (
    pd.Series(common_users)
    .sample(
        n=min(100, len(common_users)),
        random_state=42
    )
    .tolist()
)

train_history = (
    train_df.groupby("author_id")["product_id"]
    .apply(list)
    .to_dict()
)

test_history = (
    test_df.groupby("author_id")["product_id"]
    .apply(set)
    .to_dict()
)

print("Common users:", len(common_users))
print("Selected evaluation users:", len(evaluation_users))


Common users: 30061
Selected evaluation users: 100


In [4]:

# Weighted Content-Based Model

def prepare_text(column):
    return products_df[column].fillna("").astype(str)

weighted_text = (
    prepare_text("product_name") + " " +
    prepare_text("brand_name") + " " +
    prepare_text("brand_name") + " " +
    prepare_text("primary_category") + " " +
    prepare_text("primary_category") + " " +
    prepare_text("primary_category") + " " +
    prepare_text("secondary_category") + " " +
    prepare_text("secondary_category") + " " +
    prepare_text("tertiary_category") + " " +
    prepare_text("highlights") + " " +
    prepare_text("ingredients")
)

weighted_vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=10000,
    min_df=2,
    ngram_range=(1, 2),
    dtype=np.float32
)

weighted_matrix = weighted_vectorizer.fit_transform(
    weighted_text
)

print("WEIGHTED CONTENT MODEL")
print("Matrix shape:", weighted_matrix.shape)


WEIGHTED CONTENT MODEL
Matrix shape: (8494, 10000)


In [5]:

# Ingredients-Focused Content Model

ingredient_text = (
    prepare_text("product_name") + " " +
    prepare_text("brand_name") + " " +
    prepare_text("primary_category") + " " +
    prepare_text("secondary_category") + " " +
    prepare_text("highlights") + " " +
    prepare_text("highlights") + " " +
    prepare_text("ingredients") + " " +
    prepare_text("ingredients")
)

ingredient_vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=10000,
    min_df=2,
    ngram_range=(1, 2),
    dtype=np.float32
)

ingredient_matrix = ingredient_vectorizer.fit_transform(
    ingredient_text
)

print("INGREDIENTS-FOCUSED MODEL")
print("Matrix shape:", ingredient_matrix.shape)


INGREDIENTS-FOCUSED MODEL
Matrix shape: (8494, 10000)


In [6]:

# General Recommendation Function

def recommend_with_matrix(user_id, matrix, top_n=5):

    liked_ids = train_history.get(str(user_id), [])

    valid_ids = [
        pid for pid in liked_ids
        if pid in product_indices.index
    ]

    if not valid_ids:
        return []

    liked_indices = [
        int(product_indices[pid])
        for pid in valid_ids
    ]

    user_profile = np.asarray(
        matrix[liked_indices].mean(axis=0),
        dtype=np.float32
    )

    scores = cosine_similarity(
        user_profile,
        matrix
    ).ravel()

    # Exclude products used in training
    scores[liked_indices] = -np.inf

    ranked_indices = np.argsort(scores)[::-1]

    selected_indices = [
        idx for idx in ranked_indices
        if np.isfinite(scores[idx])
    ][:top_n]

    return products_df.iloc[
        selected_indices
    ]["product_id"].tolist()


In [7]:

# Evaluation Metrics

def evaluate_model(matrix, users, k=5):

    results = []

    for user_id in users:

        recommended = recommend_with_matrix(
            user_id,
            matrix,
            top_n=k
        )

        relevant = test_history.get(str(user_id), set())

        if not relevant:
            continue

        hits = [
            int(pid in relevant)
            for pid in recommended
        ]

        precision = sum(hits) / k

        recall = sum(hits) / len(relevant)

        cumulative_hits = 0
        precision_sum = 0

        for rank, hit in enumerate(hits, start=1):
            if hit:
                cumulative_hits += 1
                precision_sum += cumulative_hits / rank

        ap = precision_sum / min(len(relevant), k)

        results.append({
            "Precision@5": precision,
            "Recall@5": recall,
            "MAP@5": ap
        })

    return pd.DataFrame(results).mean()


In [8]:

# Compare Recommendation Models

baseline_results = evaluate_model(
    baseline_matrix,
    evaluation_users
)

weighted_results = evaluate_model(
    weighted_matrix,
    evaluation_users
)

ingredient_results = evaluate_model(
    ingredient_matrix,
    evaluation_users
)

comparison_df = pd.DataFrame({
    "Baseline TF-IDF": baseline_results,
    "Weighted Content": weighted_results,
    "Ingredients Focused": ingredient_results
}).T

print("MODEL COMPARISON RESULTS")
print("-" * 40)

display(comparison_df.round(4))


MODEL COMPARISON RESULTS
----------------------------------------


,Precision@5,Recall@5,MAP@5
Baseline TF-IDF,0.056,0.1812,0.1321
Weighted Content,0.058,0.1829,0.1294
Ingredients Focused,0.056,0.1812,0.1294


In [9]:

# Find the best model based on MAP@5

best_model_name = comparison_df["MAP@5"].idxmax()

print("BEST MODEL:", best_model_name)

print("\nBest model performance:")
display(comparison_df.loc[[best_model_name]])

model_matrices = {
    "Baseline TF-IDF": baseline_matrix,
    "Weighted Content": weighted_matrix,
    "Ingredients Focused": ingredient_matrix
}

best_matrix = model_matrices[best_model_name]


BEST MODEL: Baseline TF-IDF

Best model performance:


,Precision@5,Recall@5,MAP@5
Baseline TF-IDF,0.056,0.18125,0.132083


In [10]:

# Compare Top 5 Recommendations

sample_user = evaluation_users[0]

print("Selected user:", sample_user)

for model_name, matrix in model_matrices.items():

    print("\nMODEL:", model_name)

    recommended_ids = recommend_with_matrix(
        sample_user,
        matrix,
        top_n=5
    )

    recommended_products = products_df[
        products_df["product_id"].isin(recommended_ids)
    ][[
        "product_id",
        "product_name",
        "brand_name",
        "primary_category"
    ]].copy()

    recommended_products["rank"] = (
        recommended_products["product_id"]
        .map({
            pid: rank
            for rank, pid in enumerate(
                recommended_ids,
                start=1
            )
        })
    )

    display(
        recommended_products.sort_values("rank")
    )


Selected user: 7252834899

MODEL: Baseline TF-IDF


,product_id,product_name,brand_name,primary_category,rank
7523,P474377,Indigo Overnight Repair Serum in Cream Treatment,Tatcha,Skincare,1
8353,P501178,YUZU-C Glow-On Shot Serum,WASO,Skincare,2
8350,P501171,YUZU-C Beauty Sleeping Mask,WASO,Skincare,3
7549,P504303,Strengthening Nighttime Ritual Set,Tatcha,Skincare,4
7548,P501323,Overnight Recovery Duo,Tatcha,Skincare,5



MODEL: Weighted Content


,product_id,product_name,brand_name,primary_category,rank
8353,P501178,YUZU-C Glow-On Shot Serum,WASO,Skincare,1
7523,P474377,Indigo Overnight Repair Serum in Cream Treatment,Tatcha,Skincare,2
8350,P501171,YUZU-C Beauty Sleeping Mask,WASO,Skincare,3
8352,P501199,YUZU-C Eye Awakening Essence,WASO,Skincare,4
8349,P501170,SHIKULIME Color Control Oil-Free Moisturizer S...,WASO,Skincare,5



MODEL: Ingredients Focused


,product_id,product_name,brand_name,primary_category,rank
8353,P501178,YUZU-C Glow-On Shot Serum,WASO,Skincare,1
7523,P474377,Indigo Overnight Repair Serum in Cream Treatment,Tatcha,Skincare,2
8350,P501171,YUZU-C Beauty Sleeping Mask,WASO,Skincare,3
8352,P501199,YUZU-C Eye Awakening Essence,WASO,Skincare,4
7549,P504303,Strengthening Nighttime Ritual Set,Tatcha,Skincare,5
